# Autopatch demo performance

Analyze the saved MultiPatch logs from an extended autopatch demo run: the
find → seal → break-in funnel, throughput/efficiency, where time goes, why
attempts fail, and the electrical quality of the cells that made it to whole-cell.

Metrics are reconstructed from the `state_change` sequence and `test_pulse`
events in each `MultiPatch_*.log` (the tidy `patchRecord` is emitted as a Qt
signal, not written to the log). Parsing/aggregation live in `autopatch_log.py`
and `autopatch_metrics.py` (unit-tested under `tests/`).

## 1. Configure & load

Point `ROOTS` at the directory (or directories) holding the demo's
`MultiPatch_*.log` files. They're found recursively, one attempt-set per file.

The analysis is restricted to attempts that actually approached a cell
(`ONLY_APPROACHED`); setup/cleaning cycles that never engaged a cell are
excluded so the funnel and active-time reflect only real patch attempts.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('.'))
import autopatch_log as al
import autopatch_metrics as am

# ---- EDIT ME: directories containing the demo's MultiPatch_*.log files ----
# One extended-demo run per root works best (throughput assumes contiguous time).
ROOTS = [
    '/home/martin/src/acq4/junk_data/2024.08.29_000',
]

# Restrict the analysis to attempts that actually engaged a cell (reached the
# approach stage). Attempts that never got past bath/clean/out are pipette
# setup and cleaning cycles, not real patch attempts -- keeping them would
# inflate the funnel base and pad the active time with pre/post-demo idle.
ONLY_APPROACHED = True

plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.grid': True,
                     'grid.alpha': 0.25, 'axes.axisbelow': True})
# Consistent funnel palette (dark→light single hue reads as a progression).
FUNNEL_COLORS = ['#1a4a73', '#2d6ca8', '#4a90d9', '#8bc0f0', '#2e8b57']
OK, BAD = '#2e8b57', '#c0392b'

def si(val, unit):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return 'n/a'
    for p, s in [(1e9, 'G'), (1e6, 'M'), (1e3, 'k'), (1, ''), (1e-3, 'm'), (1e-6, 'µ'), (1e-9, 'n'), (1e-12, 'p')]:
        if abs(val) >= p:
            return f'{val / p:.2f} {s}{unit}'
    return f'{val:.2e} {unit}'

In [ ]:
# raw_attempts keeps every attempt (incl. setup/cleaning idle) for the timeline;
# `attempts` is the analysis population -- approached attempts only by default.
raw_attempts = al.load_run(ROOTS)
attempts = al.approached_attempts(raw_attempts) if ONLY_APPROACHED else raw_attempts
df = am.attempts_to_dataframe(attempts)
assert len(df), 'No approached attempts found. Check ROOTS, or set ONLY_APPROACHED=False.'
n_excluded = len(raw_attempts) - len(attempts)
print(f'{len(al.find_logs(ROOTS))} log files, {len(raw_attempts)} raw attempts, '
      f'{len(df)} approached (analyzed), {n_excluded} setup/cleaning excluded, '
      f"{df['device'].nunique()} device(s)")
df.head()

## 2. Headline throughput & efficiency

In [ ]:
tp = am.throughput(df)
print('Logs                :', int(tp['n_logs']))
print('Approached attempts :', int(tp['n_attempts']))
print('Cells found         :', int(tp['n_found']))
print('Seals formed        :', int(tp['n_sealed']))
print('Whole-cell recordings:', int(tp['n_whole_cell']))
print('Overall yield       : {:.1f} %  (whole-cell / approached)'.format(tp['overall_yield_pct']))
print('Active time         : {:.2f} h  (demo window, summed per-log)'.format(tp['active_hours']))
print('Attempts / hour     : {:.1f}'.format(tp['attempts_per_hour']))
print('Whole-cells / hour  : {:.2f}'.format(tp['whole_cells_per_hour']))
print('Mean attempt length : {:.1f} min'.format(tp['mean_attempt_minutes']))

## 3. The funnel: approached → found → sealed → whole-cell

How many *approached* attempts survive each stage, and the conversion rate
between stages. Attempts that never engaged a cell (setup/cleaning cycles) are
excluded, so the base of the funnel is the cells we actually tried to patch.

In [ ]:
funnel = am.funnel_counts(df)
labels = ['Approached', 'Found\ncell', 'Sealed', 'Whole\ncell']
colors = [FUNNEL_COLORS[1], FUNNEL_COLORS[2], FUNNEL_COLORS[3], FUNNEL_COLORS[4]]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, funnel['count'], color=colors, width=0.72)
for bar, (_, row) in zip(bars, funnel.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{int(row['count'])}\n{row['pct_of_approached']:.0f}%",
            ha='center', va='bottom', fontsize=10)
    if row['stage'] not in ('approached',):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
                f"→{row['conversion_from_prev']:.0f}%", ha='center', va='center',
                color='white', fontsize=9, fontweight='bold')
ax.set_ylabel('approached attempts')
ax.set_title('Autopatch funnel (bar height = count, mid-bar = conversion from previous stage)')
ax.margins(y=0.15)
plt.tight_layout(); plt.show()
funnel

## 4. Throughput over the run

Cumulative whole-cell recordings against elapsed time — the slope is the
effective yield rate.

In [ ]:
cum = am.cumulative_whole_cells(df)
fig, ax = plt.subplots()
if len(cum):
    ax.step(cum['minutes'], cum['cumulative_whole_cells'], where='post', color=OK, lw=2)
    ax.scatter(cum['minutes'], cum['cumulative_whole_cells'], color=OK, zorder=3)
ax.set_xlabel('minutes since first attempt')
ax.set_ylabel('cumulative whole-cell recordings')
ax.set_title('Whole-cell yield over the run')
plt.tight_layout(); plt.show()

## 5. Where the time goes

Per-attempt duration, and total time spent in each patch state across the run.

In [ ]:
dwell = am.state_dwell_times(attempts)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.hist(df['duration_s'] / 60.0, bins=20, color=FUNNEL_COLORS[1], edgecolor='white')
ax1.set_xlabel('attempt duration (min)'); ax1.set_ylabel('attempts')
ax1.set_title('Per-attempt duration')

ax2.barh(dwell['state'], dwell['total_s'] / 60.0, color=FUNNEL_COLORS[2])
ax2.invert_yaxis()
ax2.set_xlabel('total minutes in state (all attempts)')
ax2.set_title('Time budget by state')
plt.tight_layout(); plt.show()
dwell

## 6. Where attempts fail

Terminal outcome of every attempt. `whole cell` is success; the rest are the
state the attempt gave up in.

In [ ]:
fm = am.failure_mode_counts(df)
colors = [OK if o == 'whole cell' else BAD for o in fm['outcome']]
fig, ax = plt.subplots()
bars = ax.bar(fm['outcome'], fm['count'], color=colors, width=0.7)
for bar, c, p in zip(bars, fm['count'], fm['pct']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{int(c)}\n{p:.0f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('attempts'); ax.set_title('Attempt outcomes')
ax.margins(y=0.15)
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()
fm

## 7. Quality of the whole-cell recordings

For attempts that broke in: peak seal resistance, and access resistance /
holding current during whole-cell (from the test pulses). Lower access
resistance and smaller holding current are healthier.

In [ ]:
wc = df[df['broke_in']].copy()
if len(wc):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].hist(wc['max_seal_resistance'].dropna() / 1e9, bins=12, color=FUNNEL_COLORS[0], edgecolor='white')
    axes[0].set_xlabel('peak seal resistance (GΩ)'); axes[0].set_title('Seal quality')
    axes[0].axvline(1.0, color=BAD, ls='--', lw=1, label='1 GΩ')
    axes[0].legend()
    axes[1].hist(wc['access_resistance'].dropna() / 1e6, bins=12, color=FUNNEL_COLORS[1], edgecolor='white')
    axes[1].set_xlabel('access resistance (MΩ)'); axes[1].set_title('Access resistance')
    axes[2].hist(wc['holding_current'].dropna() / 1e-12, bins=12, color=FUNNEL_COLORS[2], edgecolor='white')
    axes[2].set_xlabel('holding current (pA)'); axes[2].set_title('Holding current')
    plt.tight_layout(); plt.show()
    print('median peak seal R :', si(wc['max_seal_resistance'].median(), 'Ω'))
    print('median access R    :', si(wc['access_resistance'].median(), 'Ω'))
    print('median holding I   :', si(wc['holding_current'].median(), 'A'))
else:
    print('No whole-cell recordings in this dataset.')

## 8. Per-attempt drill-down

One row per attempt with outcome and key numbers. Sort/filter as needed — e.g.
`table[table['found_cell'] & ~table['broke_in']]` to inspect attempts that found
a cell but never reached whole-cell.

In [ ]:
table = df[['cell_dir', 'device', 'attempt_index', 'best_stage_name', 'outcome',
            'duration_s', 'max_seal_resistance', 'access_resistance',
            'holding_current', 'n_test_pulses']].copy()
table['duration_min'] = (table.pop('duration_s') / 60.0).round(1)
for col, unit in [('max_seal_resistance', 'Ω'), ('access_resistance', 'Ω'), ('holding_current', 'A')]:
    table[col] = table[col].map(lambda v: si(v, unit))
table

## 9. Full timeline per folder

One Gantt chart per analyzed folder: every device's patch states over wall-clock,
using **all** attempts (including the setup/cleaning idle that the rest of the
analysis excludes). The shaded span is the analyzed demo window — first to last
approached attempt — so you can see the idle before and after that is trimmed
from the active time.

In [ ]:
tl = am.state_timeline(raw_attempts)

# Analyzed demo window per folder = span of the approached attempts we keep.
window_by_folder = {}
for a in attempts:
    f = os.path.dirname(a.source)
    lo, hi = window_by_folder.get(f, (a.start_time, a.end_time))
    window_by_folder[f] = (min(lo, a.start_time), max(hi, a.end_time))

# Stable color per state across all folders.
states_seen = list(dict.fromkeys(tl['state']))
cmap = plt.get_cmap('tab20')
state_color = {s: cmap(i % 20) for i, s in enumerate(states_seen)}

for folder, g in tl.groupby('folder'):
    t0 = g['t_start'].min()
    devices = sorted(g['device'].unique())
    ypos = {d: i for i, d in enumerate(devices)}
    fig, ax = plt.subplots(figsize=(12, 0.6 * len(devices) + 2))
    if folder in window_by_folder:
        lo, hi = window_by_folder[folder]
        ax.axvspan((lo - t0) / 60.0, (hi - t0) / 60.0, color='0.5', alpha=0.15,
                   zorder=0, label='analyzed demo window')
    for _, r in g.iterrows():
        width = max((r['t_end'] - r['t_start']) / 60.0, 1e-3)
        ax.broken_barh([((r['t_start'] - t0) / 60.0, width)],
                       (ypos[r['device']] - 0.4, 0.8),
                       facecolors=state_color[r['state']], edgecolor='none')
    ax.set_yticks(range(len(devices)))
    ax.set_yticklabels(devices)
    ax.set_ylim(-0.6, len(devices) - 0.4)
    ax.set_xlabel('minutes since first event in folder')
    ax.set_title(f'{os.path.basename(folder)}  —  shaded = analyzed demo window')
    ax.grid(True, axis='x', alpha=0.25)
    handles = [plt.Rectangle((0, 0), 1, 1, color=state_color[s]) for s in states_seen]
    ax.legend(handles, states_seen, ncol=min(len(states_seen), 6), fontsize=7,
              loc='upper center', bbox_to_anchor=(0.5, -0.3), frameon=False)
    plt.tight_layout()
    plt.show()